<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/04_tool_surface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 — The tool surface, measured

**The claim you should be able to make when you finish:** *"The tool surface is
a design artifact with measurable properties. I lint mine in CI, and I know what
it costs me per turn and what it costs me in selection accuracy."*

Lab 2 told you what a tool costs (its schema is resent every turn). Lab 3 told
you what a wrong call costs (it is a failed step, and steps compound). This lab
is about the surface itself — the thing that, in most real agents, accumulated
rather than being designed.

Two facts sit behind almost every "the model is being stupid" report:

1. schemas are resent on **every** request;
2. selection accuracy falls as the surface grows, and it falls fastest between
   tools that *look alike* — the model is doing fuzzy matching over names and
   descriptions, and three near-identical strings is exactly what fuzzy matching
   gets wrong.

Twenty-five minutes, no API key.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 1. A surface that accumulated

This is not a strawman. Every one of these patterns comes from a real API that
grew a tool at a time, each addition locally reasonable.

In [ ]:
from agentlab.loop import Tool

surface = [
    Tool("get_status", "Get status."),
    Tool("fetch_status", "Fetch the status of a job."),
    Tool("query_status", "Query the status for a given job id."),
    Tool("search_orders",
         "Free-text search across every order in the account and return all matching results.",
         {"type": "object", "properties": {"q": {"type": "string"},
                                           "sort": {"type": "string"}}}),
    Tool("delete_order", "Permanently delete an order.", read_only=True, destructive=True),
    Tool("read_customer_record", "Read a customer's private account details from the database."),
]

for t in surface:
    print(f"  {t.name:<22} {t.description}")

## 2. Lint it

None of the findings below are style points. Each one is a failure you would
otherwise diagnose from a confused transcript at 11pm.

In [ ]:
from agentlab.tools import report

print(report(surface))

Walk them:

- **`get_status` / `fetch_status` / `query_status`** — same object,
  interchangeable verb. There is no signal that separates them. Selection here is
  close to a coin flip, and a coin flip is a failed step 50% of the time.
- **`search_orders` has no `limit`** — a tool that *can* return an unbounded blob
  will one day return one, into a transcript that is resent every turn
  afterwards. That is lab 2's quadratic with a much bigger `g`.
- **`sort` is free text** where a fixed set was meant. An invalid value is a
  wasted turn.
- **`delete_order` is destructive and marked read-only** — the harness has no
  hook to gate it. It cannot gate what it cannot name.

The important property: **this runs with no model, no API key and no network**.
Put it in CI.

In [ ]:
from agentlab.tools import lint

def gate(tools):
    blocking = [f for f in lint(tools) if f.severity == "high"]
    if blocking:
        raise AssertionError("tool surface has blocking findings:\n"
                             + "\n".join(str(f) for f in blocking))
    print("tool surface is clean")

try:
    gate(surface)
except AssertionError as exc:
    print(f"CI would fail here:\n{exc}")

## 3. Fix it, and watch the findings go

The fix for an ambiguous pair is not a better description of each. It is
**fewer tools**: merge them, and make the one that survives say when *not* to use
it.

In [ ]:
fixed = [
    Tool("get_job_status",
         "Return the current status of one job by id. Use this for a job you already "
         "have an id for. To find a job, use search_orders instead.",
         {"type": "object", "properties": {"job_id": {"type": "string"}}, "required": ["job_id"]}),
    Tool("search_orders",
         "Free-text search across orders. Returns at most `limit` summaries, newest first; "
         "call get_order for the full record.",
         {"type": "object", "properties": {
             "q": {"type": "string", "description": "Text to match."},
             "sort": {"type": "string", "enum": ["newest", "oldest", "value"]},
             "limit": {"type": "integer", "default": 20}},
          "required": ["q"]}),
    Tool("delete_order", "Permanently delete an order. Cannot be undone.",
         {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
         read_only=False, destructive=True),
    Tool("read_customer_record", "Read a customer's private account details from the database.",
         {"type": "object", "properties": {"customer_id": {"type": "string"}},
          "required": ["customer_id"]}),
]

print(report(fixed))
gate(fixed)

Six tools became four, every finding cleared, and the surface got *cheaper* per
turn as well as clearer. That direction is typical: the fixes for accuracy and
the fixes for cost are usually the same fixes.

## 4. What surface size costs

The token cost is the easy half to measure, and it is charged on every request.

In [ ]:
from agentlab.budget import context_share
from agentlab.tools import distractors, surface_tokens

print(f"{'tools':>6} {'tokens/request':>15} {'per tool':>10} {'% window':>10} {'over 40 turns':>15}")
for n in (4, 10, 30, 60, 150):
    tokens = surface_tokens(distractors(n))
    print(f"{n:>6} {tokens:>15,} {tokens // n:>10} {context_share(tokens):>9.2%} {tokens * 40:>15,}")

print("\nNote the per-tool figure: about 74 tokens. Lab 2's planning heuristic")
print("was 380, and both are honest — these generated schemas are terse, and a")
print("real MCP tool with prose descriptions and eight typed parameters is not.")
print("A 5x spread is why you measure your own surface instead of using anyone's")
print("rule of thumb, including this repo's.")

That per-tool spread is worth dwelling on. Schema verbosity is a real lever and
an unglamorous one: descriptions written for a human reader, `$ref`-expanded
enums, and every optional parameter documented in full will quietly triple your
fixed prefix. Count it once, with the real tokenizer, and you will know whether
it is worth an afternoon.

The harder half is selection accuracy, and it is not something this repo can
measure for you offline — it needs a model. What you *can* do offline is measure
the thing that drives it: how many tools in your surface are hard to tell apart.

Below, a realistic back-office surface grows, and we count the confusable pairs.

In [ ]:
print(f"{'tools':>6} {'ambiguous pairs':>17} {'high-severity findings':>24}")
for n in (10, 20, 40, 80, 150):
    findings = lint(distractors(n), budget_share=1.0)
    pairs = [f for f in findings if f.kind == "ambiguous_pair"]
    high = [f for f in findings if f.severity == "high"]
    print(f"{n:>6} {len(pairs):>17} {len(high):>24}")

print("\nCollisions grow faster than the surface: more tools means more chances")
print("that two of them share a noun, and the model is matching on the noun.")

### What to do when you genuinely have 150 tools

You do not have to choose between "all the schemas, all the time" and "fewer
features". Three real options, in the order they usually apply:

1. **Namespace and merge.** Most 150-tool surfaces are 40 tools with variants.
   This is the cheapest fix and it is a refactor, not an architecture change.
2. **Defer loading (tool search).** Keep the schemas out of the prefix and let
   the model search for what it needs. Overhead then grows with tools *used*
   rather than tools *available*. Note that it appends rather than swaps, which
   is what makes it cache-safe — see lab 2 on why that matters.
3. **Split across subagents.** Each one gets a small, coherent surface. This is
   lab 8, and it costs you the reliability arithmetic there.

## 5. Errors are prompts

This is the highest-leverage and least-loved part of tool design.

A tool error is the only text in the loop written specifically for a model that
has **just made a mistake and is about to decide what to do next**. Treat it as
UI copy for an audience of one.

In [ ]:
from agentlab.tools import error

bad = "Error: 400"
good = error(
    "search_orders failed",
    "`sort` must be one of the supported values (got 'recent')",
    "Retry with a valid sort value, or omit it for the default",
    valid=["newest", "oldest", "value"],
)
print("what most tools return:\n ", bad)
print("\nwhat it should return:\n ", good)
print(f"\ncost of the difference: about {len(good) // 4 - len(bad) // 4} tokens.")
print("value of the difference: one turn — and a turn resends the whole transcript.")

Three things a recoverable error message contains:

1. **what** failed (the tool name — the model may have made several calls);
2. **why**, specifically, including the offending value;
3. **what to do next** — and, where the space is small, the valid set.

The anti-pattern is an error that is accurate and useless: `ValidationError:
invalid input`. The model's only remaining move is to guess.

## 6. The measurement you should actually run

Everything above is static. The dynamic version needs a model and about twenty
minutes: take fifty real requests, run them against your surface at 10 tools and
at 50 tools, and count how often the *right* tool was called first.

The harness for it is lab 6's eval kit, and `Case(requires=...)` is the check —
a trajectory assertion, not an outcome one. That is the right shape: you are
measuring selection, not the final answer.

In [ ]:
from agentlab.evals import Case

selection_cases = [
    Case("sel-1", "What's the status of job 4471?", requires=("get_job_status",), max_steps=3),
    Case("sel-2", "Find the orders from last week over $500", requires=("search_orders",), max_steps=3),
    Case("sel-3", "What's on file for customer c-99?", requires=("read_customer_record",),
         forbids=("delete_order",), max_steps=3),
]
for case in selection_cases:
    print(f"  {case.id}: requires {case.requires}, forbids {case.forbids or '()'}")
print("\nRun these against 4 tools and against 4 + 50 distractors. The delta is your answer.")

## What you can now say

- *"The tool surface is a design artifact. I lint mine in CI, with no model in
  the loop."*
- *"`get_status` and `fetch_status` in one surface is a coin flip — and a coin
  flip is a failed step half the time."*
- *"Thirty tools is 11K tokens of every single request, and 456K across a
  40-turn run."*
- *"A tool that can return an unbounded blob eventually will, into a transcript
  that's resent every turn."*
- *"An error message is a prompt for a model that just made a mistake. Ours say
  what failed, why, and what to do next."*

## Next

**[Lab 5](05_the_doom_loop.ipynb)** — what it looks like when this all goes
wrong at once.